# Shared dataset builder

Shared builder for the per-source dataset folders.

Each folder has the folder notebook with a list CONFIG of dicts:
  key, title, doi, setting, country,
  source: ('pmc', PMCID) | ('url', direct-download URL) | ('local', path)  -> where the raw file comes from
  file: raw file name (saved in <folder>/raw/), sheet: sheet name (optional), header: header row index (optional)
  label: column holding the Toxoplasma IgG result, pos / neg: label values (lower-case strings) meaning positive / negative
  keep: columns to keep even if their name looks Toxoplasma-related (e.g. partner's serostatus = an exposure)
  drop: extra columns to remove (leakage, identifiers, outcomes measured after infection)
  prep: optional function(DataFrame) -> DataFrame applied right after reading
  note: free text
run(here, CONFIG) builds processed/<key>.csv, evaluates each with protocol.evaluate, writes results.csv and DATASETS.md.

Loaded after `protocol.ipynb` with `%run -i datasets/common/folder_tools.ipynb`.

In [ ]:
import io, json, os, re, shutil, zipfile
import numpy as np
import pandas as pd
import requests

# clean_features, drop_toxo_columns, evaluate, summarise come from protocol.ipynb (loaded first with %run -i)
S3 = 'https://pmc-oa-opendata.s3.amazonaws.com'
HARVEST = os.path.join('datasets', '_harvest')     # paths are relative to the repository root

**`fetch`**

In [ ]:
def fetch(cfg, raw):
    os.makedirs(raw, exist_ok=True)
    path = os.path.join(raw, cfg['file'])
    if os.path.exists(path):
        return path
    kind, ref = cfg['source']
    if kind == 'pmc':
        local = os.path.join(HARVEST, 'supp', ref, cfg['file'])
        if os.path.exists(local):
            shutil.copy(local, path); return path
        r = requests.get(S3, params={'list-type': 2, 'prefix': ref + '.'}, timeout=60).text
        keys = [k for k in re.findall(r'<Key>([^<]+)</Key>', r) if k.endswith('/' + cfg['file'])]
        url = f'{S3}/{sorted(keys)[-1]}'
    elif kind == 'local':
        local = ref if os.path.isabs(ref) else os.path.join(HARVEST, ref)
        shutil.copy(local, path); return path
    else:
        url = ref
    r = requests.get(url, timeout=300); r.raise_for_status()
    open(path, 'wb').write(r.content)
    return path

**`_frames`**

In [ ]:
def _frames(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in ('.xlsx', '.xls', '.xlsm', '.ods'):
        return pd.read_excel(path, sheet_name=None, header=None)
    if ext in ('.csv', '.tsv', '.txt'):
        return {'csv': pd.read_csv(path, header=None, sep=None, engine='python', on_bad_lines='skip', encoding_errors='ignore')}
    if ext in ('.sav', '.por', '.dta'):
        import pyreadstat
        rd = {'.sav': pyreadstat.read_sav, '.por': pyreadstat.read_por, '.dta': pyreadstat.read_dta}[ext]
        d, _ = rd(path, apply_value_formats=True)
        return {'data': pd.concat([pd.DataFrame([d.columns], columns=d.columns), d], ignore_index=True)}
    raise ValueError(ext)

**`_header_row`**

In [ ]:
def _header_row(raw):
    best, br = -1e9, 0
    for i in range(min(8, len(raw))):
        r = raw.iloc[i]
        score = sum(isinstance(v, str) and not re.fullmatch(r'[\d.,\s-]+', v) for v in r) - 0.5 * r.isna().sum()
        if score > best:
            best, br = score, i
    return br

**`read_table`**

In [ ]:
def read_table(path, sheet=None, header=None):
    frames = _frames(path)
    raw = frames[sheet] if sheet is not None else next(iter(frames.values()))
    hr = _header_row(raw) if header is None else header
    cols, seen = [], {}
    for c in raw.iloc[hr]:
        c = re.sub(r'\s+', ' ', str(c)).strip(); seen[c] = seen.get(c, 0) + 1
        cols.append(c if seen[c] == 1 else f'{c}.{seen[c]}')
    d = raw.iloc[hr + 1:].copy(); d.columns = cols
    return d.dropna(how='all').reset_index(drop=True)

**`raw_path`**

In [ ]:
def raw_path(cfg, here):
    path = fetch(cfg, os.path.join(here, 'raw'))
    if cfg.get('member'):                      # a file inside a zip supplement
        out = os.path.join(here, 'raw', os.path.basename(cfg['member']))
        if not os.path.exists(out):
            open(out, 'wb').write(zipfile.ZipFile(path).read(cfg['member']))
        return out
    return path

**`build`**

In [ ]:
def build(cfg, here):
    if cfg.get('loader'):                      # curated loader (scripts/*_datasets.py): returns X, y
        fetch(cfg, os.path.join(here, 'raw'))
        X, y = cfg['loader']()
        X, leak = drop_toxo_columns(pd.DataFrame(X).reset_index(drop=True), keep=list(pd.DataFrame(X).columns))
        X = clean_features(X); y = pd.Series(np.asarray(y).astype(int))
        os.makedirs(os.path.join(here, 'processed'), exist_ok=True)
        X.assign(target_toxo_pos=y.values).to_csv(os.path.join(here, 'processed', f'{cfg["key"]}.csv'), index=False)
        return X, y, leak
    d = read_table(raw_path(cfg, here), cfg.get('sheet'), cfg.get('header'))
    if cfg.get('prep'):
        d = cfg['prep'](d)
    lab = d[cfg['label']].astype(str).str.strip().str.lower().str.replace(r'\.0$', '', regex=True)
    pos, neg = {str(v).lower() for v in cfg['pos']}, {str(v).lower() for v in cfg['neg']}
    y = lab.map(lambda v: 1.0 if v in pos else (0.0 if v in neg else np.nan))
    d, y = d[y.notna()], y[y.notna()].astype(int)
    X = d.drop(columns=[cfg['label']] + [c for c in cfg.get('drop', []) if c in d.columns])
    X, leak = drop_toxo_columns(X, keep=cfg.get('keep', ()))
    X = clean_features(X)
    os.makedirs(os.path.join(here, 'processed'), exist_ok=True)
    X.assign(target_toxo_pos=y.values).to_csv(os.path.join(here, 'processed', f'{cfg["key"]}.csv'), index=False)
    return X, y, leak

**`run`**

In [ ]:
def run(here, CONFIG, only=None):
    rows = []
    for cfg in CONFIG:
        if only and cfg['key'] not in only:
            continue
        try:
            X, y, leak = build(cfg, here)
            res = evaluate(X, y)
            rows.append(summarise(cfg['key'], X, y, res, dict(
                title=cfg.get('title', ''), setting=cfg.get('setting', ''), country=cfg.get('country', ''),
                doi=cfg.get('doi', ''), features=', '.join(X.columns)[:400], removed_as_leakage=', '.join(leak)[:300],
                note=cfg.get('note', ''))))
        except Exception as e:
            rows.append(dict(dataset=cfg['key'], title=cfg.get('title', ''), best_model=f'ERROR: {type(e).__name__}: {e}'[:200]))
        r = rows[-1]
        print(f"{r['dataset']:<34} n={r.get('n', '-')!s:>6} pos={r.get('positives', '-')!s:>5} feats={r.get('n_features', '-')!s:>3} "
              f"best AUC={r.get('best_auc', '-')} ({r.get('best_model', '')})")
    res = pd.DataFrame(rows)
    out = os.path.join(here, 'results.csv')
    if only and os.path.exists(out):                 # partial run: update those rows, keep the others, keep CONFIG order
        old = pd.read_csv(out)
        res = pd.concat([old[~old.dataset.isin(res.dataset)], res], ignore_index=True)
        order = {c['key']: i for i, c in enumerate(CONFIG)}
        res = res.sort_values('dataset', key=lambda s: s.map(order)).reset_index(drop=True)
    res.to_csv(out, index=False)
    write_md(here, res)
    return res

**`write_md`**

In [ ]:
def write_md(here, res):
    lines = ['| dataset | setting | country | n | positive | features | LogReg | RF | XGBoost | best |', '|---|---|---|---|---|---|---|---|---|---|']
    for _, r in res.iterrows():
        f = lambda k: '' if pd.isna(r.get(k)) else r.get(k)
        lines.append(f"| {r['dataset']} | {f('setting')} | {f('country')} | {f('n')} | {f('positives')} | {f('n_features')} | "
                     f"{f('LogReg_auc')} | {f('RandomForest_auc')} | {f('XGBoost_auc')} | **{f('best_auc')}** |")
    open(os.path.join(here, 'DATASETS.md'), 'w').write('\n'.join(lines) + '\n')